# Pixel covariance visualization
This notebook loads the Jacobian, covariance and index files produced by COLMAP's `covariance_exporter` or the `BACovariance` API. It then computes the 2x2 pixel covariance for all observations in a given image and visualises its magnitude.

In [ ]:
from pathlib import Path
import numpy as np
from scipy import sparse
import matplotlib.pyplot as plt
import pycolmap

In [ ]:
# Paths to the exported files
data_dir = Path("/path/to/project")
jac_path = data_dir / "jacobian.txt"
cov_path = data_dir / "covariance.txt"
index_path = data_dir / "index.txt"
obs_index_path = data_dir / "obs_index.txt"
model_path = data_dir / "sparse/optimized"
image_id = 5  # image to analyse

In [ ]:
def read_sparse_matrix(path):
    with open(path) as f:
        rows, cols = map(int, f.readline().split())
        data, r, c = [], [], []
        for line in f:
            rr, cc, val = line.split()
            r.append(int(rr))
            c.append(int(cc))
            data.append(float(val))
    return sparse.csr_matrix((data, (r, c)), shape=(rows, cols))


def read_dense_matrix(path):
    return np.loadtxt(path)


def read_index(path):
    mapping = {}
    with open(path) as f:
        for line in f:
            name, start, size = line.split()
            mapping[name] = (int(start), int(size))
    return mapping


def read_obs_index(path):
    entries = []
    with open(path) as f:
        for line in f:
            iid, idx, pid, row = map(int, line.split())
            entries.append((iid, idx, pid, row))
    return entries

In [ ]:
J = read_sparse_matrix(jac_path)
C = read_dense_matrix(cov_path)
index = read_index(index_path)
obs_info = read_obs_index(obs_index_path)
rec = pycolmap.Reconstruction(model_path)
img = rec.images[image_id]
cam = rec.cameras[img.camera_id]
height, width = cam.height, cam.width

In [ ]:
# Append pixel coordinates from the reconstruction
obs_info = [
    (iid, idx, pid, row, rec.images[iid].points2D[idx].xy)
    for iid, idx, pid, row in obs_info
]

In [ ]:
vals = []
coords = []
for iid, idx, pid, row_start, xy in obs_info:
    if iid != image_id:
        continue
    rows = slice(row_start, row_start + 2)
    cols = []
    rot_s, rot_sz = index.get(f"pose_{iid}_rot", (0, 0))
    trans_s, trans_sz = index.get(f"pose_{iid}_trans", (0, 0))
    cols += list(range(rot_s, rot_s + rot_sz))
    cols += list(range(trans_s, trans_s + trans_sz))
    pt_key = f"point_{pid}"
    if pt_key in index:
        ps, psz = index[pt_key]
        if ps + psz <= C.shape[0]:
            cols += list(range(ps, ps + psz))
    J_block = J[rows, :][:, cols].toarray()
    C_sub = C[np.ix_(cols, cols)]
    cov = J_block @ C_sub @ J_block.T
    val = np.linalg.norm(cov, "fro")
    vals.append(val)
    coords.append(xy)

In [ ]:
vals = np.array(vals)
print("min", vals.min(), "max", vals.max(), "mean", vals.mean())

In [ ]:
img_cov = np.zeros((height, width), dtype=float)
for (x, y), v in zip(coords, vals):
    xi, yi = int(round(x)), int(round(y))
    if 0 <= yi < height and 0 <= xi < width:
        img_cov[yi, xi] = v
plt.imshow(img_cov, cmap="inferno")
plt.colorbar(label="pixel covariance magnitude")
plt.title(f"Image {image_id}")
plt.show()